# 07 — Results: comparison of all detectors and recommendation

**Goal**: put the five families side by side under the same protocol, measure how much four failures allow us to conclude, and recommend a detector.

**Key findings**

- **34 detector configurations** from five families were run under one protocol; **18 detect all four failures** at a threshold calibrated on training data only.
- **No family dominates.** The best statistical, classical ML, time series and deep learning detectors all reach 4/4 with 0.3–0.8 false alarms per week (the expert rule: 1.1). The lowest false-alarm rate is a one-feature statistical score (robust z-score of the 2-hour mean oil temperature against the previous 7 days, **0.28 per week**); the earliest warning comes from a CUSUM chart on the motor current (**failure 4 flagged 12.9 h early**), matched by LOF and a cleaned Matrix Profile.
- **Four failures do not allow a finer ranking.** 4/4 detections are compatible with a true detection rate as low as 40% (95% interval), and the false-alarm intervals of the five best detectors overlap. Only coarse differences are established, e.g. the oil score raises significantly fewer false alarms than the expert rule (95% intervals 0.08–0.71 vs. 0.75–1.95 per week), and part of the best score's lead is a selection effect: the luckiest of 34 equivalent detectors would look just as good.
- **The training data matter more than the algorithm.** Unreported leak-like episodes in the training period blind every method that learns "normal" by nearest neighbours or reconstruction (Matrix Profile, autoencoders: 0–2/4) until they are removed with the expert rule, and give LOF a success it loses once they are removed.
- **Complexity buys nothing here.** The two statistical detectors run in under half a second, need no training data curation, are deterministic and explainable; deep autoencoders take up to a minute per model, depend on the seed and on a cleaned training set, and give no early warning.
- **Recommendation**: monitor the compressor with the CUSUM chart on motor current for early warning, and confirm with the drift-adjusted oil score. Together they detect 4/4 with 0.71 false alarms per week and a 12.9-hour warning on failure 4 (a combination defined after seeing the results, so this figure is optimistic).

In [ ]:
import os
import time
import warnings
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy import stats

from metrobench import baselines as bl
from metrobench import deep
from metrobench import evaluation as ev
from metrobench import ml
from metrobench import timeseries as ts
from metrobench.data import FAILURES, failure_mask, load, sustained_load_episodes
from metrobench.features import load_features
from metrobench.plotstyle import INK_MUTED, SERIES, use_style

use_style()
torch.set_num_threads(4)
FIG, RESULTS = Path("docs/figures"), Path("results")
grid, X = load_features()
X_train, _ = ev.split(X)
grid_train, _ = ev.split(grid)
episodes = sustained_load_episodes(load())
unreported = episodes[~episodes.reported]
train_episodes = episodes[episodes.start < ev.SPLIT]

## 1. All detectors

Results of notebooks 03–06, all on the test period (April 11 – September 1) with thresholds calibrated on the training period (≤ 1 alarm per week). For stochastic detectors, the median over seeds.

In [ ]:
cols = ["detected", "false_alarms_per_week", "lead_h_4"]
stat = pd.read_csv(RESULTS / "03_statistical.csv", index_col=0)[cols].assign(family="statistical", seeds=1)
stat.loc[stat.index.str.startswith("Sustained-load rule"), "family"] = "expert rule"
ml_runs = pd.read_csv(RESULTS / "04_classical_ml_runs.csv")
ml_med = ml_runs.groupby(["detector", "input"])[cols].median()
ml_med.index = [f"{d}, {i} features" for d, i in ml_med.index]
ml_med = ml_med.assign(family="classical ML", seeds=ml_runs.groupby(["detector", "input"]).size().to_numpy())
tsr = pd.read_csv(RESULTS / "05_time_series.csv", index_col=0)[cols].assign(family="time series", seeds=1)
dl_runs = pd.read_csv(RESULTS / "06_deep_learning_runs.csv")
dl_med = dl_runs.groupby(["model", "data"])[cols].median()
dl_med.index = [f"{m}, {d} training data" for m, d in dl_med.index]
dl_med = dl_med.assign(family="deep learning", seeds=dl_runs.groupby(["model", "data"]).size().to_numpy())
everything = pd.concat([stat, ml_med, tsr, dl_med])
everything = everything.sort_values(["detected", "false_alarms_per_week"], ascending=[False, True])
everything.to_csv(RESULTS / "07_all_detectors.csv")
print(f"{len(everything)} detector configurations, {int((everything.detected == 4).sum())} detect all four failures")
everything.round(2)

In [ ]:
FAMILIES = ["expert rule", "statistical", "classical ML", "time series", "deep learning"]
COLORS = dict(zip(FAMILIES, [INK_MUTED, SERIES[0], SERIES[2], SERIES[1], SERIES[6]]))
full = everything[everything.detected == 4]
fig, ax = plt.subplots(figsize=(10, 5))
for fam in FAMILIES:
    g = full[full.family == fam]
    ax.scatter(g.false_alarms_per_week, g.lead_h_4, s=70, color=COLORS[fam], edgecolor="white", linewidth=1, label=fam, zorder=3)
labels = {"Rolling-baseline z, oil 2 h": (8, 4), "CUSUM, motor current": (8, 2), "LOF, raw features": (8, -10),
          "Sustained-load rule (expert, 0.9)": (10, -14), "Matrix Profile, load state, z-norm, cleaned ref.": (8, -4),
          "Dense AE, cleaned training data": (-150, -14)}
for name, (dx, dy) in labels.items():
    if name in full.index:
        r = full.loc[name]
        ax.annotate(name, (r.false_alarms_per_week, r.lead_h_4), xytext=(dx, dy), textcoords="offset points", fontsize=7, color=INK_MUTED)
ax.axhline(0, color=INK_MUTED, lw=0.8)
ax.set_ylim(-2, 16)
ax.set_xlabel("false alarms per week (test period, calibrated threshold)")
ax.set_ylabel("early warning on failure 4 (hours before reported start)")
ax.set_xscale("log")
ticks = [0.25, 0.5, 1, 2, 4, 8]
ax.set_xticks(ticks, [f"{t:g}" for t in ticks])
ax.minorticks_off()
ax.legend(loc="lower right", fontsize=8)
ax.set_title("Detectors that find all four failures: fewer false alarms (left) and earlier warning (top) are better")
fig.savefig(FIG / "07_summary.png")
plt.show()

In [ ]:
counts = everything.groupby("family").agg(configurations=("detected", "size"), detect_all_four=("detected", lambda s: int((s == 4).sum())))
counts.reindex(FAMILIES)

Every family has at least one configuration that detects the four failures, but they get there differently. In the statistical family, the winners follow one physically meaningful signal and either accumulate evidence (CUSUM) or compare with the recent past (rolling reference). In the classical ML family, LOF wins on the raw window features. The time series and deep learning families only reach 4/4 once leak-like episodes are removed from their reference. The forecast residuals reach 4/4 too, but with 2–6 false alarms per week, which no maintenance team would accept.

## 2. How much can four failures tell?

The test period contains four failures and about 15 weeks of data. Both numbers are small, so every metric carries a large uncertainty. The best detectors are re-scored here to get their exact alarm counts.

In [ ]:
top = {
    "Rolling-baseline z, oil 2 h": bl.rolling_baseline_zscore(X.oil_mean_2h),
    "CUSUM, motor current": bl.cusum(grid.Motor_current, grid_train.Motor_current, k=0.5),
    "LOF, raw features (seed 0)": ml.local_outlier_factor(X, X_train, seed=0),
    "Matrix Profile, load state, cleaned ref.": ts.matrix_profile(grid.DV_eletric, ev.SPLIT, exclude=train_episodes),
    "Dense AE, cleaned training data (seed 0)": deep.dense_autoencoder(X, X_train[~failure_mask(X_train.index, train_episodes)], seed=0)[0],
    "Sustained-load rule (expert, 0.9)": X.load_share_30min,
}
fixed = {"Sustained-load rule (expert, 0.9)": 0.9}
rows, events = [], {}
for name, score in top.items():
    score_train, score_test = ev.split(score)
    threshold = fixed.get(name, ev.calibrate(score_train))
    r = ev.evaluate(score_test, threshold)
    weeks = score_test.notna().sum() / (7 * 24 * 60)
    n = r["false_alarms"]
    lo, hi = (stats.chi2.ppf(0.025, 2 * n) / 2 if n else 0.0), stats.chi2.ppf(0.975, 2 * n + 2) / 2
    events[name] = r["events"]
    rows.append({"detector": name, "detected": r["detected"], "false alarms": n, "weeks scored": round(weeks, 1),
                 "per week": n / weeks, "95% CI low": lo / weeks, "95% CI high": hi / weeks,
                 "lead F4 (h)": r["lead_hours"].get(4, np.nan)})
uncertainty = pd.DataFrame(rows).set_index("detector")
uncertainty.round(2)

In [ ]:
lo, hi = stats.beta.ppf(0.025, 4, 1), 1.0
print(f"4 detections out of 4: 95% Clopper-Pearson interval for the detection rate = [{lo:.2f}, {hi:.2f}]")
print(f"3 detections out of 4: [{stats.beta.ppf(0.025, 3, 2):.2f}, {stats.beta.ppf(0.975, 4, 1):.2f}]")

In [ ]:
# Selection effect: the best of 34 detectors with the same true false-alarm rate still looks better than it is
rng = np.random.default_rng(0)
best = [rng.poisson(0.7 * 15, size=34).min() / 15 for _ in range(10_000)]
print(f"34 detectors, each with a true rate of 0.70 false alarms per week over 15 weeks: "
      f"the best one shows {np.mean(best):.2f} per week on average")

Three kinds of uncertainty limit the conclusions.

- **Detection rate.** With four failures, 4/4 is compatible with a true detection rate anywhere between 40% and 100%, and 3/4 with 19% to 99%. A detector that "only" found 3/4 cannot be called worse than one that found 4/4 on this evidence.
- **False alarm rate.** False alarms are rare events over 14–16 weeks; their 95% Poisson intervals are wide. The oil score (4 false alarms, 0.08–0.71 per week) is significantly better than the expert rule (19, 0.75–1.95), but the intervals of the oil score, CUSUM, LOF, the cleaned Matrix Profile and the cleaned dense autoencoder all overlap.

- **Selection.** The best of 34 configurations benefits from being the best: among 34 detectors that all had a true rate of 0.70 false alarms per week, the luckiest would show 0.29 on average, almost exactly the oil score's 0.28. The configurations are not identical, so this does not show that the oil score is average, but its lead over the others could be largely a selection effect.

The honest summary is therefore a short list of equally good detectors rather than a single winner, with the choice among them made on other criteria: early warning, robustness, cost and explainability.

## 3. Robustness to the alarm budget

The calibration budget (≤ 1 alarm per week on training data) is a choice. A maintenance team could accept more alarms, or demand fewer. The deterministic top detectors are recalibrated with three budgets:

In [ ]:
budgets = [0.5, 1.0, 2.0]
rows = []
for name in ["Rolling-baseline z, oil 2 h", "CUSUM, motor current", "Matrix Profile, load state, cleaned ref."]:
    score_train, score_test = ev.split(top[name])
    for b in budgets:
        r = ev.evaluate(score_test, ev.calibrate(score_train, max_alarms_per_week=b))
        rows.append({"detector": name, "training budget (alarms/week)": b, "detected": r["detected"],
                     "false alarms / week (test)": round(r["false_alarms_per_week"], 2),
                     "lead F4 (h)": round(r["lead_hours"].get(4, np.nan), 1)})
pd.DataFrame(rows).set_index(["detector", "training budget (alarms/week)"])

CUSUM on the motor current is the most robust: 4/4 and a 12–13.5-hour warning on failure 4 for every budget, with false alarms between 0.58 and 0.83 per week. The oil score is the most frugal but also the most sensitive to the budget: at 0.5 alarm per week on training data it drops to 3/4. The cleaned Matrix Profile keeps 4/4 everywhere, but its early warning on failure 4 only appears at the loosest budget.

## 4. Cost of complexity

In [ ]:
def timed(fn):
    t0 = time.perf_counter(); fn(); return time.perf_counter() - t0

cost = pd.DataFrame({
    "fit + score (s)": [
        timed(lambda: bl.rolling_baseline_zscore(X.oil_mean_2h)),
        timed(lambda: bl.cusum(grid.Motor_current, grid_train.Motor_current)),
        timed(lambda: ml.local_outlier_factor(X, X_train, seed=0)),
        timed(lambda: ts.matrix_profile(grid.DV_eletric, ev.SPLIT, exclude=train_episodes)),
        timed(lambda: deep.dense_autoencoder(X, X_train[~failure_mask(X_train.index, train_episodes)], seed=0)),
        timed(lambda: deep.lstm_autoencoder(grid, ev.SPLIT, exclude=train_episodes, seed=0)),
    ],
    "inputs": ["1 feature", "1 signal", "18 features", "1 signal", "18 features", "5 signals"],
    "needs a cleaned reference": ["no", "no", "no (but relies on contamination)", "yes", "yes", "yes"],
    "result varies with seed": ["no", "no", "yes (0.35–1.17 FA/week)", "no", "yes", "yes (2–4/4)"],
    "explains its alarms": ["yes (one signal)", "yes (one signal)", "no", "partly (matching subsequence)", "per-feature error", "per-signal error"],
}, index=["Rolling-baseline z, oil", "CUSUM, motor current", "LOF", "Matrix Profile, load state", "Dense AE", "LSTM AE"])
cost["fit + score (s)"] = cost["fit + score (s)"].round(1)
cost

## 5. Recommendation

In [ ]:
def union(*names):
    merged = pd.concat([events[n][["start", "end"]] for n in names]).sort_values("start")
    alarms = pd.Series(False, index=ev.split(X)[1].index)
    for s, e in merged.itertuples(index=False):
        alarms.loc[s:e] = True
    return alarms.astype(float).where(ev.split(top[names[0]])[1].notna() | ev.split(top[names[1]])[1].notna())

two_tier = union("CUSUM, motor current", "Rolling-baseline z, oil 2 h")
r = ev.evaluate(two_tier, 1.0)
pd.Series({"detected": f"{r['detected']}/4", "false alarms / week": round(r["false_alarms_per_week"], 2),
           "lead F4 (h)": round(r["lead_hours"][4], 1),
           "leads (h)": {k: round(v, 1) for k, v in r["lead_hours"].items()}}).to_frame("CUSUM current OR rolling oil z")

The CUSUM chart and the drift-adjusted oil score are complementary: CUSUM accumulates the small rise of motor current that precedes failure 4, the oil score raises few false alarms. Raising an alarm when either fires gives the table above: **4/4, 0.71 false alarms per week (about three a month), failure 4 flagged 12.9 h before its reported start, the others within 1.6 h after**.

This combination was chosen after seeing the test results, so its figures are optimistic; it should be validated on new failures before deployment. It is still the recommended setup, for reasons that do not depend on the test period:

- both components are **deterministic** (no seed), need **no curated training set**, and run in under a second for seven months of data;
- each alarm points to **one physical signal** (motor current or oil temperature) that a technician can check directly;
- CUSUM is the most **robust** detector to the choice of alarm budget.

The machine learning and deep learning detectors are not worth their cost on this machine. They become worth revisiting with more data: several trains, more failures, and other failure types than air leaks, where a hand-picked signal may not exist.

## 6. Limitations

- **Four failures, one train, one type of fault.** All conclusions rest on four air leaks of one compressor. Rankings within the top group are not statistically established, and nothing here says how the detectors would handle other faults.
- **Incomplete labels.** Eight leak-like episodes are missing from the failure reports. They count as false alarms in this benchmark, so false-alarm rates are upper bounds, and the reference cleaning relies on the expert rule rather than on ground truth.
- **One chronological split.** A single training period (February – April 10) is used. A rolling evaluation (retraining every month and scoring the next) would give more test events and a better view of drift.
- **Choices made after seeing results.** Parameters were fixed in advance and every variant tried is reported, but some decisions (the 12-hour detection window, the two-tier combination, the drift adjustment) were informed by earlier notebooks on the same test period.
- **Next steps**: apply the protocol to other trains or periods (for instance the MetroPT data described by Veloso et al., 2022), evaluate with a rolling split, and test supervised early-warning models that use the precursors seen before failure 4.